In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import logging

LOGGER = logging.getLogger(__name__)

logging.basicConfig(level=logging.CRITICAL, format="%(name)s %(asctime)s %(message)s")
LOGGER.setLevel(logging.INFO)

In [3]:
from utils.database_utils import generate_database_and_retriever, populate_database
from scipy.spatial.distance import cosine

In [4]:
from utils.node_standarization import translate_nodes

In [5]:
data_base = "./localdb"
retriever = generate_database_and_retriever(main_folder=data_base)

all_keys = list(retriever.docstore.yield_keys())
all_documents = retriever.docstore.mget(all_keys)
docutments_dic = {all_keys[i]: all_documents[i] for i in range(len(all_keys))}

In [6]:
import pickle

with open("graph_raw_data.pkl", "rb") as f:
    graph_elements = pickle.load(f)

In [7]:
graph_elements_translated = translate_nodes(graph_elements)

In [8]:
from utils.node_standarization import lemmatize_nodes_and_relationships

graph_elements_lematized = lemmatize_nodes_and_relationships(graph_elements_translated)

In [64]:
from langchain_ollama import OllamaEmbeddings, OllamaLLM
from langchain_core.messages import SystemMessage, HumanMessage
from typing import List
from collections import Counter


class EmbeddingNode:
    model_name: str

    def __init__(self, model_name):
        self.model = OllamaEmbeddings(model=model_name)

    def get_embedding(self, text):
        return self.model.embed_query(text)


class DescriptionNode:
    def __init__(self, model_name):
        self.model = OllamaLLM(model=model_name)
        self.system_instruction = """
            You are an expert Graph Data Analyst. Your task is to extract a simple description that summarizes of all the provided descriptions. 
            ### Output Format
            The output should be a single string that summarizes all the provided descriptions.
        """
        self.prompt = """
            Descriptions: {descriptions}
        """

    def get_description(self, text):
        formatted_prompt = self.prompt.format(descriptions=text)
        messages = [
            SystemMessage(content=self.system_instruction),
            HumanMessage(content=formatted_prompt),
        ]
        summary = self.model.invoke(messages)
        return summary


class Node:
    embedding: List[int]
    name: str
    type: str
    description: str
    model: EmbeddingNode
    node_id: str = None

    def __init__(self, name, type, description, model, node_id):
        self.name = name
        self.type = type
        self.description = description
        self.model = model
        self.node_id = node_id

    def get_embedding(self):
        self.embedding = self.model.get_embedding(self.name)


class RepresentativeNode:
    group_id: str
    mapping_nodes: List[str]
    description: str = None
    name: str = None
    type: str = None

    def __init__(self, group_id, all_nodes, model, existing_nodes):
        self.group_id = group_id
        group_nodes = [node for node in all_nodes if node.group_id == group_id]

        if (
            existing_node := self.check_if_nodes_are_related_to_existing_nodes(
                group_nodes, existing_nodes
            )
        ) is not None:
            LOGGER.info(
                "Found existing node for group id: {}".format(existing_node.name)
            )
            self.description = existing_node.description
            self.name = existing_node.name
            self.type = existing_node.type

        else:
            LOGGER.info("No existing node found for group id: {}".format(group_id))
            all_descriptions = [node.description for node in group_nodes]
            unique_descriptions = set(all_descriptions)
            if len(unique_descriptions) == 1:
                self.description = list(unique_descriptions)[0]

            else:
                self.description = model.get_description(
                    "\n".join(list(unique_descriptions))
                )

            type = Counter(node.type for node in group_nodes)
            self.type = type.most_common(1)[0][0]

            name = Counter(node.name for node in group_nodes)
            self.name = name.most_common(1)[0][0]

        self.mapping_nodes = [
            node.node_id for node in group_nodes
        ]  ## This will need to be the node id

    def check_if_nodes_are_related_to_existing_nodes(
        self, nodes, existing_nodes, threshold=0.05
    ):
        for node in nodes:
            for existing_node in existing_nodes:
                if cosine(node.embedding, existing_node.embedding) < threshold:
                    return existing_node

        return None


In [23]:
import tqdm
from neo4j import GraphDatabase

In [55]:
URI = "bolt://localhost:7687"
AUTH = ("neo4j", "123456789")


def get_current_nodes():
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        current_nodes = []
        for node in driver.session().run("MATCH (n) RETURN n"):
            current_nodes.append(node["n"])
        return current_nodes


embedding_model = EmbeddingNode(model_name="embeddinggemma:latest")


def from_results_to_nodes(results):
    nodes = []
    for res in results:
        try:
            name = res._properties["name"]
            description = res._properties["description"]
            type = list(res.labels)[0]
            nodes.append(Node(name, type, description, embedding_model, None))
        except Exception:
            LOGGER.info("Node {} not an entity".format(res._properties["name"]))

    for node in tqdm.tqdm(nodes):
        node.get_embedding()
    return nodes


In [56]:
existing_nodes = get_current_nodes()
existing_nodes = from_results_to_nodes(existing_nodes)

__main__ 2026-03-24 19:42:12,960 Node 698e779b472793e10560333a3fb982dc911195548bbe5b732033d4ad2fc4cf48 not an entity
__main__ 2026-03-24 19:42:12,961 Node e57660b2511d9dea96431d00135106b4482c8f83430bb4b60927a78c5458fb12 not an entity
__main__ 2026-03-24 19:42:12,961 Node 7169bbbf01be31cd11e4082f5590edd47bf0c47defb693ecb93978a5b7ae810e not an entity
__main__ 2026-03-24 19:42:12,961 Node 4dfee41dfbf9fe62df5de7b2c496f5bd35c791535e61058ba1797d047642699c not an entity
__main__ 2026-03-24 19:42:12,962 Node ce200155a612cbf84a93fe777599b60b7bb8605349cfeff0104d5a81d3c452b4 not an entity
__main__ 2026-03-24 19:42:12,962 Node 10818432f2614e7b12a484ed31623db1506e333ad468f80dd8bee12ef95a5006 not an entity
__main__ 2026-03-24 19:42:12,963 Node fdae9f4bdbd291e2489f6f89ffa245e6f29c2716841dd4d9fc02e81efdcfcaca not an entity
__main__ 2026-03-24 19:42:12,963 Node 7809fc8b2a4a5b18ec0ddcfca7a768cf8ef9c0acf62a152b81ab13e7721a6ac9 not an entity
__main__ 2026-03-24 19:42:12,964 Node ad4f23557b79a1869da0347cd4

In [57]:
import enum

from pyvis import node


all_nodes = []
embedding_model = EmbeddingNode(model_name="embeddinggemma:latest")
for document_id, relations in tqdm.tqdm(
    graph_elements_lematized.items(), total=len(graph_elements_lematized)
):
    for index_relation, relationship in enumerate(relations):
        head_id = document_id + "_" + "head" + "_" + str(index_relation)
        tail_id = document_id + "_" + "tail" + "_" + str(index_relation)
        node_head = Node(
            name=relationship.head,
            type=relationship.head_type,
            description=relationship.head_description,
            model=embedding_model,
            node_id=head_id,
        )
        relationship.head_id = head_id
        node_head.get_embedding()
        all_nodes.append(node_head)

        node_tail = Node(
            name=relationship.tail,
            type=relationship.tail_type,
            description=relationship.tail_description,
            model=embedding_model,
            node_id=tail_id,
        )
        relationship.tail_id = tail_id
        node_tail.get_embedding()
        all_nodes.append(node_tail)


100%|██████████| 9/9 [00:09<00:00,  1.09s/it]


In [58]:
class UnionFind:
    def __init__(self, nodes):
        self.parent = {node.id: node.id for node in nodes}

    def find(self, i):
        if self.parent[i] == i:
            return i
        self.parent[i] = self.find(self.parent[i])
        return self.parent[i]

    def union(self, i, j):
        root_i = self.find(i)
        root_j = self.find(j)
        if root_i != root_j:
            self.parent[root_i] = root_j


# 1. Ensure initial state is None
for node in all_nodes:
    node.group_id = None

for index, node in enumerate(all_nodes):
    node.id = "node_" + str(index)

uf = UnionFind(all_nodes)
threshold = 0.05

# 2. Perform the comparisons
for i in range(len(all_nodes)):
    for j in range(i + 1, len(all_nodes)):
        node1, node2 = all_nodes[i], all_nodes[j]

        if cosine(node1.embedding, node2.embedding) < threshold:
            uf.union(node1.id, node2.id)

# 3. Final Assignment
# This converts the internal 'parent' pointers into a final, shared ID
for node in all_nodes:
    node.group_id = uf.find(node.id)

nodes_by_group = {}
for node in all_nodes:
    group_id = node.group_id
    if group_id not in nodes_by_group:
        nodes_by_group[group_id] = []
    nodes_by_group[group_id].append(node)


In [65]:
model_for_descriptions = DescriptionNode(model_name="gemma3:12b")
nodes_representatives = []
for group_id, nodes in tqdm.tqdm(nodes_by_group.items(), total=len(nodes_by_group)):
    nodes_representatives.append(
        RepresentativeNode(group_id, nodes, model_for_descriptions, existing_nodes)
    )

  0%|          | 0/60 [00:00<?, ?it/s]__main__ 2026-03-24 19:45:08,704 Found existing node for group id: analysis
__main__ 2026-03-24 19:45:08,709 Found existing node for group id: model classification error
__main__ 2026-03-24 19:45:08,714 Found existing node for group id: methodology
__main__ 2026-03-24 19:45:08,716 Found existing node for group id: two interpretable surrogate model
__main__ 2026-03-24 19:45:08,718 Found existing node for group id: linear regression model
__main__ 2026-03-24 19:45:08,721 Found existing node for group id: interpretable surrogate model
__main__ 2026-03-24 19:45:08,725 Found existing node for group id: decision tree model
__main__ 2026-03-24 19:45:08,727 Found existing node for group id: original model
__main__ 2026-03-24 19:45:08,730 Found existing node for group id: optimal threshold
__main__ 2026-03-24 19:45:08,732 Found existing node for group id: 0.48
__main__ 2026-03-24 19:45:08,732 Found existing node for group id: 0.36
__main__ 2026-03-24 19:45:

# Reconstruc the clean graph

In [68]:
for node_representative in nodes_representatives:
    for document_id, relations in graph_elements_lematized.items():
        for index_relation, relationship in enumerate(relations):
            if relationship.head_id in node_representative.mapping_nodes:
                relationship.head = node_representative.name
                relationship.head_type = node_representative.type
                relationship.head_description = node_representative.description
            if relationship.tail_id in node_representative.mapping_nodes:
                relationship.tail = node_representative.name
                relationship.tail_type = node_representative.type
                relationship.tail_description = node_representative.description

In [69]:
import pickle

with open("graph_clean_data.pkl", "wb") as f:
    pickle.dump(graph_elements_lematized, f)